# 🚀 Fundamentos de LLM para Developers & QA

Este notebook está preparado para explicar fundamentos de LLMs y además conectar con **OpenAI**, **Claude** y **Ollama**.

## Librerías consideradas desde tu `pyproject.toml`
- `openai>=1.109.1`
- `anthropic>=0.69.0`
- `ollama>=0.6.0`
- `tiktoken>=0.11.0`
- `plotly>=6.3.0`
- `numpy>=2.3.3`
- `litellm>=1.77.5`

## Objetivo
Entender cómo funciona un LLM por dentro, cómo se controla, y dejar una base práctica para demos con distintos proveedores.

> **Por default queda OpenAI**, pero puedes cambiar fácilmente el proveedor en una sola celda.


## 🧠 Introducción

### Frontier models
Son los modelos más avanzados disponibles hoy.  
Están en el límite actual de lo que la IA puede hacer en lenguaje, razonamiento, herramientas, visión y generación de código.

Ejemplos:
- GPT
- Claude
- Gemini

### Open Source
Modelos que puedes descargar o correr localmente:
- LLaMA
- Mistral
- DeepSeek
- Ollama *(Ollama no es un modelo: es una herramienta para correr modelos localmente)*

### Transformers
Los LLM modernos están construidos sobre la arquitectura Transformer.

Idea simple:
- reciben texto
- analizan relaciones entre palabras
- ponen atención a distintas partes del input
- predicen la siguiente salida probable

La pieza clave es **attention**.

### Parámetros
Son los pesos internos del modelo.

En simple:
- más parámetros → más capacidad para representar patrones complejos
- pero más parámetros no siempre significa mejor respuesta para todos los casos


In [36]:
import os
import numpy as np
import plotly.graph_objects as go

from openai import OpenAI
from anthropic import Anthropic
import ollama

print("Imports listos ✅")


Imports listos ✅


In [37]:
# =========================
# Configuración del proveedor
# =========================

ACTIVE_PROVIDER = "openai"   # Cambia aquí: "openai", "claude", "ollama"

MODEL_CONFIG = {
    "openai": "gpt-4.1-mini",
    "claude": "claude-3-5-haiku-latest",
    "ollama": "llama3.2"
}

ACTIVE_MODEL = MODEL_CONFIG[ACTIVE_PROVIDER]

print("Proveedor activo:", ACTIVE_PROVIDER)
print("Modelo activo:", ACTIVE_MODEL)


Proveedor activo: openai
Modelo activo: gpt-4.1-mini


In [38]:
# =========================
# Clientes
# =========================

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY")) if os.getenv("OPENAI_API_KEY") else None
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY")) if os.getenv("ANTHROPIC_API_KEY") else None

def provider_ready(provider):
    if provider == "openai":
        return openai_client is not None
    if provider == "claude":
        return anthropic_client is not None
    if provider == "ollama":
        return True
    return False

print("OpenAI listo:", provider_ready("openai"))
print("Claude listo:", provider_ready("claude"))
print("Ollama listo:", provider_ready("ollama"))


OpenAI listo: True
Claude listo: True
Ollama listo: True


In [39]:
# =========================
# Función unificada para texto
# =========================

def generate_text(
    prompt,
    system_prompt="Eres un asistente claro y técnico.",
    temperature=0.2,
    top_p=1.0,
    max_output_tokens=150,
    provider=ACTIVE_PROVIDER,
    model=None,
):
    model = model or MODEL_CONFIG[provider]

    if provider == "openai":
        if openai_client is None:
            raise ValueError("No se encontró OPENAI_API_KEY.")
        response = openai_client.responses.create(
            model=model,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt},
            ],
            temperature=temperature,
            top_p=top_p,
            max_output_tokens=max_output_tokens,
        )
        return {
            "provider": provider,
            "model": model,
            "text": response.output_text,
            "usage": getattr(response, "usage", None),
            "raw": response,
        }

    elif provider == "claude":
        if anthropic_client is None:
            raise ValueError("No se encontró ANTHROPIC_API_KEY.")
        response = anthropic_client.messages.create(
            model=model,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_output_tokens,
        )
        text = "".join(
            block.text for block in response.content if getattr(block, "type", "") == "text"
        )
        return {
            "provider": provider,
            "model": model,
            "text": text,
            "usage": {
                "input_tokens": getattr(response.usage, "input_tokens", None),
                "output_tokens": getattr(response.usage, "output_tokens", None),
            } if getattr(response, "usage", None) else None,
            "raw": response,
        }

    elif provider == "ollama":
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt},
            ],
            options={
                "temperature": temperature,
                "top_p": top_p,
                "num_predict": max_output_tokens,
            }
        )
        text = response["message"]["content"]
        return {
            "provider": provider,
            "model": model,
            "text": text,
            "usage": None,
            "raw": response,
        }

    else:
        raise ValueError("Proveedor no soportado: " + str(provider))


In [40]:
# Prueba rápida de conectividad
# Si no quieres consumir tokens todavía, puedes no ejecutar esta celda.

if provider_ready(ACTIVE_PROVIDER):
    result = generate_text("Explica qué es un LLM en una frase.")
    print("Proveedor:", result["provider"])
    print("Modelo:", result["model"])
    print("Respuesta:", result["text"])
    print("Uso:", result["usage"])
else:
    print("El proveedor no está listo. Revisa credenciales o que Ollama esté levantado.")


Proveedor: openai
Modelo: gpt-4.1-mini
Respuesta: Un LLM (Large Language Model) es un modelo de inteligencia artificial entrenado con grandes cantidades de texto para comprender y generar lenguaje natural de manera coherente y contextual.
Uso: ResponseUsage(input_tokens=31, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=35, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=66)


## Celda 1 — ¿Qué es OpenAI?

**OpenAI es una empresa.**

- OpenAI → la empresa
- GPT → la familia de modelos
- ChatGPT → el producto

### Idea clave
> Empresa ≠ modelo ≠ producto


## Celda 2 — ¿Qué es un LLM realmente?

**LLM** significa **Large Language Model**.

En palabras simples:
> es un modelo entrenado con muchísimo texto para predecir la siguiente palabra o fragmento más probable.

### Idea clave
> Un LLM no adivina por magia. Calcula probabilidades.


In [41]:
words = ["azul", "gris", "rojo", "verde", "negro"]
probs = np.array([0.62, 0.18, 0.08, 0.07, 0.05], dtype=float)
probs = probs / probs.sum()

fig = go.Figure(
    data=[go.Bar(x=words, y=probs, text=[f"{p:.0%}" for p in probs], textposition="outside")]
)

fig.update_layout(
    title="Siguiente palabra probable para: 'El cielo es de color...'",
    xaxis_title="Palabra candidata",
    yaxis_title="Probabilidad",
    yaxis=dict(range=[0, 0.75]),
)

fig.show()
print("Palabra más probable:", words[int(np.argmax(probs))])


Palabra más probable: azul


## Celda 3 — ¿Qué son los tokens?

Un **token** es una unidad pequeña de texto que el modelo realmente procesa.

### ¿Por qué importa?
- el costo normalmente se calcula por tokens
- el contexto tiene límite de tokens
- el rendimiento cambia según cuántos tokens envías y recibes

### Tokenizer oficial de OpenAI
https://platform.openai.com/tokenizer


In [42]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4")

texts = [
    "Hola, estoy aprendiendo LLMs.",
    "Tokenizar no es lo mismo que contar palabras.",
    "Developers y QA necesitan entender costo y contexto."
]

for text in texts:
    token_ids = encoding.encode(text)
    print("=" * 80)
    print("Texto:")
    print(text)
    print("\nToken IDs:")
    print(token_ids)
    print("Cantidad de tokens:", len(token_ids))


Texto:
Hola, estoy aprendiendo LLMs.

Token IDs:
[69112, 11, 82384, 68446, 37116, 445, 11237, 82, 13]
Cantidad de tokens: 9
Texto:
Tokenizar no es lo mismo que contar palabras.

Token IDs:
[3404, 11403, 912, 1560, 781, 36994, 1744, 89805, 74304, 13]
Cantidad de tokens: 10
Texto:
Developers y QA necesitan entender costo y contexto.

Token IDs:
[21076, 388, 379, 67008, 25203, 13145, 96537, 78949, 379, 77843, 13]
Cantidad de tokens: 11


## Celda 4 — ¿Qué es el contexto?

El **contexto** es toda la información que el modelo puede ver en una interacción.

### Idea clave
> Un usuario conversa. Un ingeniero administra contexto.

La IA muchas veces no “recuerda”; solo parece recordar porque tú sigues enviando historial.


In [10]:
conversation_with_history = [
    {"role": "user", "content": "Mi nombre es Rager."},
    {"role": "assistant", "content": "Entendido, tu nombre es Rager."},
    {"role": "user", "content": "¿Cuál es mi nombre?"}
]

conversation_without_history = [
    {"role": "user", "content": "¿Cuál es mi nombre?"}
]

print("Conversación CON historial:")
for msg in conversation_with_history:
    print(f"- {msg['role']}: {msg['content']}")

print("\nConversación SIN historial:")
for msg in conversation_without_history:
    print(f"- {msg['role']}: {msg['content']}")

print("\nConclusión: si no mandas el historial, el modelo no tiene por qué saber el nombre.")


Conversación CON historial:
- user: Mi nombre es Rager.
- assistant: Entendido, tu nombre es Rager.
- user: ¿Cuál es mi nombre?

Conversación SIN historial:
- user: ¿Cuál es mi nombre?

Conclusión: si no mandas el historial, el modelo no tiene por qué saber el nombre.


In [44]:
# Ejemplo opcional con proveedor real

if provider_ready(ACTIVE_PROVIDER):
    result_followup = generate_text(
        prompt="¿Cuál es mi nombre?",
        system_prompt="Responde solo con el nombre."
    )
    print(result_followup["text"])
else:
    print("Proveedor no listo.")


No tengo esa información.


## Celda 5 — ¿Qué es un prompt?

Un **prompt** es la instrucción que le das al modelo.

### Ejemplo de prompt débil
`Explícame IA`

### Ejemplo de prompt mejor diseñado
`Explícame qué es un LLM en 3 puntos, nivel principiante, con un ejemplo pequeño y lenguaje simple.`


In [12]:
prompt_debil = "Explícame IA"
prompt_fuerte = (
    "Explícame qué es un LLM en 3 puntos, "
    "nivel principiante, con lenguaje simple y un ejemplo práctico."
)

print("Prompt débil:")
print(prompt_debil)
print("\nPrompt fuerte:")
print(prompt_fuerte)


Prompt débil:
Explícame IA

Prompt fuerte:
Explícame qué es un LLM en 3 puntos, nivel principiante, con lenguaje simple y un ejemplo práctico.


In [45]:
if provider_ready(ACTIVE_PROVIDER):
    print("=== Respuesta a prompt débil ===")
    weak = generate_text(prompt_debil)
    print(weak["text"])

    print("\n=== Respuesta a prompt fuerte ===")
    strong = generate_text(prompt_fuerte)
    print(strong["text"])
else:
    print("Proveedor no listo.")


=== Respuesta a prompt débil ===
Claro, te explico qué es la Inteligencia Artificial (IA) de manera clara y técnica.

**Definición:**
La Inteligencia Artificial es una rama de la informática que se enfoca en crear sistemas y programas capaces de realizar tareas que normalmente requieren inteligencia humana. Estas tareas incluyen el razonamiento, aprendizaje, reconocimiento de patrones, comprensión del lenguaje natural, percepción visual, toma de decisiones, entre otras.

**Componentes clave de la IA:**

1. **Aprendizaje Automático (Machine Learning):**  
   Es una subárea de la IA que permite a las máquinas aprender a partir de datos sin ser programadas explícitamente para cada tarea. Utiliza algoritmos que identifican patrones y hacen predicciones o decisiones bas

=== Respuesta a prompt fuerte ===
Claro, aquí tienes una explicación sencilla sobre qué es un LLM:

1. **Qué es un LLM:**  
   Un LLM (Large Language Model) es un programa de computadora que ha aprendido a entender y genera

## Celda 6 — ¿Qué son los roles? (`system`, `user`, `assistant`)

- `system` → reglas y comportamiento
- `user` → lo que la persona pide
- `assistant` → la respuesta del modelo

### Idea clave
> El rol más poderoso es `system`.


In [46]:
messages = [
    {"role": "system", "content": "Eres un arquitecto de software. Responde claro y en español."},
    {"role": "user", "content": "Explícame qué es un LLM en 2 puntos."},
    {"role": "assistant", "content": "Un LLM es un modelo de lenguaje entrenado con mucho texto."},
    {"role": "user", "content": "Ahora compáralo con una API tradicional."}
]

for msg in messages:
    print(f"[{msg['role'].upper()}]")
    print(msg["content"])
    print()


[SYSTEM]
Eres un arquitecto de software. Responde claro y en español.

[USER]
Explícame qué es un LLM en 2 puntos.

[ASSISTANT]
Un LLM es un modelo de lenguaje entrenado con mucho texto.

[USER]
Ahora compáralo con una API tradicional.



In [48]:
if provider_ready(ACTIVE_PROVIDER):
    result = generate_text(
        prompt="Explícame qué es API en 2 puntos.",
        system_prompt="Eres un profesor claro, técnico y breve. Responde en español."
    )
    print(result["text"])
else:
    print("Proveedor no listo.")


1. **Definición:** API (Interfaz de Programación de Aplicaciones) es un conjunto de reglas y protocolos que permite que diferentes programas o sistemas se comuniquen entre sí.

2. **Función:** Facilita la integración y el intercambio de datos o funcionalidades entre aplicaciones, sin que el usuario vea el funcionamiento interno.


## Celda 7 — ¿Qué es `temperature`?

`temperature` controla qué tan **creativa o variada** será la salida.

- baja → más predecible
- alta → más variación


In [49]:
base_words = ["azul", "gris", "rojo", "verde", "morado"]
base_probs = np.array([0.60, 0.20, 0.08, 0.07, 0.05], dtype=float)

def apply_temperature(probabilities, temperature):
    probabilities = np.array(probabilities, dtype=float)
    logits = np.log(probabilities + 1e-12)
    adjusted = np.exp(logits / temperature)
    adjusted = adjusted / adjusted.sum()
    return adjusted

temp_low = apply_temperature(base_probs, 0.4)
temp_high = apply_temperature(base_probs, 1.5)

fig_low = go.Figure(data=[go.Bar(x=base_words, y=temp_low, text=[f"{p:.0%}" for p in temp_low], textposition="outside")])
fig_low.update_layout(title="Temperature baja (0.4)", yaxis=dict(range=[0, 1]))
fig_low.show()

fig_high = go.Figure(data=[go.Bar(x=base_words, y=temp_high, text=[f"{p:.0%}" for p in temp_high], textposition="outside")])
fig_high.update_layout(title="Temperature alta (1.5)", yaxis=dict(range=[0, 1]))
fig_high.show()


In [52]:
if provider_ready(ACTIVE_PROVIDER):
    print("=== Temperature baja ===")
    low = generate_text(
        prompt="Dame un slogan para una cafetería.",
        temperature=0.1
    )
    print(low["text"])

    print("\n=== Temperature alta ===")
    high = generate_text(
        prompt="Dame un slogan para una cafetería.",
        temperature=0.9
    )
    print(high["text"])
else:
    print("Proveedor no listo.")


=== Temperature baja ===
Claro, aquí tienes un slogan para una cafetería:

**"Tu momento perfecto, taza a taza."** 

¿Quieres que sea más formal, divertido o creativo?

=== Temperature alta ===
Claro, aquí tienes un slogan para una cafetería:

**"Tu momento, nuestro café."** 

¿Quieres algo más cálido, moderno o tradicional?


## Celda 8 — ¿Qué es `top_p`?

`top_p` controla cuántas opciones probables entran al juego.

- bajo → menos opciones
- alto → más opciones


In [53]:
candidate_words = ["azul", "gris", "rojo", "verde", "morado", "blanco"]
candidate_probs = np.array([0.50, 0.20, 0.12, 0.08, 0.06, 0.04], dtype=float)

def top_p_filter(words, probabilities, top_p=0.8):
    pairs = sorted(zip(words, probabilities), key=lambda x: x[1], reverse=True)
    selected_words = []
    selected_probs = []
    cumulative = 0.0
    for word, prob in pairs:
        selected_words.append(word)
        selected_probs.append(prob)
        cumulative += prob
        if cumulative >= top_p:
            break
    selected_probs = np.array(selected_probs, dtype=float)
    selected_probs = selected_probs / selected_probs.sum()
    return selected_words, selected_probs

selected_words, selected_probs = top_p_filter(candidate_words, candidate_probs, top_p=0.80)

print("Palabras seleccionadas con top_p=0.80:", selected_words)
print("Probabilidades re-normalizadas:", selected_probs)

fig = go.Figure(data=[go.Bar(x=selected_words, y=selected_probs, text=[f"{p:.0%}" for p in selected_probs], textposition="outside")])
fig.update_layout(title="Opciones consideradas después de aplicar top_p=0.80", yaxis=dict(range=[0, 1]))
fig.show()


Palabras seleccionadas con top_p=0.80: ['azul', 'gris', 'rojo']
Probabilidades re-normalizadas: [0.6097561  0.24390244 0.14634146]


In [54]:
if provider_ready(ACTIVE_PROVIDER):
    print("=== top_p bajo ===")
    low = generate_text(
        prompt="Escribe una frase creativa sobre inteligencia artificial.",
        top_p=0.3,
        temperature=0.7
    )
    print(low["text"])

    print("\n=== top_p alto ===")
    high = generate_text(
        prompt="Escribe una frase creativa sobre inteligencia artificial.",
        top_p=1.0,
        temperature=0.7
    )
    print(high["text"])
else:
    print("Proveedor no listo.")


=== top_p bajo ===
La inteligencia artificial es el puente invisible que conecta la imaginación humana con el poder infinito de los datos.

=== top_p alto ===
La inteligencia artificial es el puente invisible que conecta la imaginación humana con el futuro digital.


## Celda 9 — ¿Qué son los `max tokens`?

`max tokens` o `max_output_tokens` controla el máximo de tokens que el modelo puede generar en la respuesta.


In [55]:
examples = [
    {"input": "Resume la arquitectura en una frase.", "max_output_tokens": 20},
    {"input": "Explica la arquitectura en detalle.", "max_output_tokens": 150},
]

for ex in examples:
    print("=" * 80)
    print("Prompt:", ex["input"])
    print("max_output_tokens:", ex["max_output_tokens"])


Prompt: Resume la arquitectura en una frase.
max_output_tokens: 20
Prompt: Explica la arquitectura en detalle.
max_output_tokens: 150


In [56]:
if provider_ready(ACTIVE_PROVIDER):
    print("=== max_output_tokens pequeño ===")
    short = generate_text(
        prompt="Explica qué es un microservicio.",
        max_output_tokens=30
    )
    print(short["text"])

    print("\n=== max_output_tokens grande ===")
    long = generate_text(
        prompt="Explica qué es un microservicio.",
        max_output_tokens=120
    )
    print(long["text"])
else:
    print("Proveedor no listo.")


=== max_output_tokens pequeño ===
Un microservicio es un enfoque arquitectónico para desarrollar aplicaciones de software donde la aplicación se divide en pequeños servicios independientes, cada uno con una funcionalidad

=== max_output_tokens grande ===
Un microservicio es un enfoque arquitectónico para desarrollar aplicaciones de software donde la funcionalidad se divide en pequeños servicios independientes y autónomos. Cada microservicio está diseñado para cumplir con una única responsabilidad o función específica dentro del sistema, y puede ser desarrollado, desplegado y escalado de manera independiente.

Características principales de un microservicio:

1. **Independencia**: Cada microservicio funciona de forma autónoma, con su propia lógica de negocio, base de datos y ciclo de vida.
2. **Despliegue independiente**: Se pueden actualizar o modificar sin afectar a


## Celda 10 — ¿Qué es el determinismo en modelos?

Buscamos que una misma entrada produzca una salida igual o muy parecida de forma consistente.

Para acercarte al determinismo:
- `temperature` baja
- `top_p` bajo o controlado
- prompts estables
- mismo modelo
- mismo contexto


In [57]:
rng_words = ["azul", "gris", "rojo", "verde"]
rng_probs = np.array([0.65, 0.20, 0.10, 0.05], dtype=float)

def sample_next_word(words, probs, seed=None):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(words), p=probs)
    return words[idx]

print("Sin seed:")
for _ in range(5):
    print(sample_next_word(rng_words, rng_probs))

print("\nCon seed fija:")
for _ in range(5):
    print(sample_next_word(rng_words, rng_probs, seed=42))


Sin seed:
azul
azul
azul
azul
azul

Con seed fija:
gris
gris
gris
gris
gris


In [58]:
if provider_ready(ACTIVE_PROVIDER):
    for i in range(3):
        result = generate_text(
            prompt="Resume qué es un LLM en una frase.",
            temperature=0.0,
            top_p=0.2,
            max_output_tokens=40
        )
        print(f"Intento {i+1}: {result['text']}")
else:
    print("Proveedor no listo.")


Intento 1: Un LLM (Large Language Model) es un modelo de inteligencia artificial entrenado con grandes cantidades de texto para comprender y generar lenguaje natural de manera coherente.
Intento 2: Un LLM (Large Language Model) es un modelo de inteligencia artificial entrenado con grandes cantidades de texto para comprender y generar lenguaje natural de manera coherente y contextual.
Intento 3: Un LLM (Large Language Model) es un modelo de inteligencia artificial entrenado con grandes cantidades de texto para comprender y generar lenguaje natural de manera coherente y contextual.


## Celda 11 — ¿Qué es LiteLLM?

**LiteLLM** es una librería open source que da una interfaz unificada para llamar a muchos proveedores de LLM.


In [59]:
from litellm import completion

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explícame qué es un LLM en una frase."}],
    mock_response="Un LLM es un modelo que predice texto basándose en patrones aprendidos."
)

print("Respuesta simulada:")
print(response.choices[0].message.content)

print("\nUso reportado:")
print(response.usage)

cost = response._hidden_params.get("response_cost", 0)

print("\nCosto estimado por LiteLLM:")
print(f"${cost:.8f}")


Respuesta simulada:
Un LLM es un modelo que predice texto basándose en patrones aprendidos.

Uso reportado:
Usage(completion_tokens=20, prompt_tokens=10, total_tokens=30, completion_tokens_details=None, prompt_tokens_details=None)

Costo estimado por LiteLLM:
$0.00001350


## ✅ Conclusión

- un LLM trabaja con **tokens**
- decide por **probabilidades**
- depende del **contexto**
- obedece mejor con un buen **prompt**
- cambia de comportamiento con `temperature`, `top_p` y `max_output_tokens`

Eso es lo que separa usar IA como usuario… de diseñarla como ingeniero.
